### BUTTER-E Feature Engineering

Builds the shared-schema feature table for BUTTER-E (MLP family), for pooling with EC-NAS later. Source: `runs_with_standardized_energy.csv` — already the filter-passed subset (confirmed: `filter == False` for all 37,055 rows, verified in `01a_butter_e_eda.ipynb`).

Decisions locked in during EDA (see `01a_butter_e_eda.ipynb` §8.0 for full reasoning):
- `flops` — approximated as `2 × params` (no per-layer widths available from `shape` name alone; upgrade to precise reconstruction later only if this approximation hurts model accuracy)
- `epochs` — fixed constant `3000` (BUTTER-E's primary sweep uses a fixed epoch budget, not early stopping; supported by tight within-config `run_time` coefficient-of-variation, not independently confirmed at the per-run level)

**Update 1, added after the `03a_within_butter_e.ipynb` diagnostic showed a within-family Kendall-Tau of only 0.274:** two columns are added here that were missing from the original 9-column core schema —

- `is_gpu` — was in the raw data (`runs_with_standardized_energy.csv` has it directly) but excluded from the processed feature table entirely. CPU vs. GPU is a large, structural difference in power draw, plausibly a major missing signal.
- `shape` (one-hot encoded, 8 categories: `rectangle`, `trapezoid`, `exponential`, `rectangle_residual`, `wide_first_2x/4x/8x/16x`) — added as a **family-specific auxiliary feature**, per the thesis's Section 3.3.2 design (masked to zero when not applicable to a family).

**Update 2, added then reverted:** `memory_fit_ratio` (params × 4 bytes ÷ node RAM, via a `node_sinfo.csv` join on `df['node']`) was tried next but confirmed *not* to help in `03a_within_butter_e.ipynb` — a small regression on every metric. **Removed** — the `node_sinfo.csv` join and this column are dropped from the pipeline.

**Update 3, added then replaced (this update):** `dataset` (one-hot, 12 categories) was promoted in after `03h_diagnostic_feature_audit.ipynb` found it was the single largest driver of BUTTER-E's energy variance, and confirmed a large win in `03a` (R² 0.075→0.973, tau 0.319→0.936). **Replaced here with `dataset` *properties* instead of `dataset` *identity*, for generalization** — one-hot identity only lets the model recognize the 12 specific PMLB datasets it was trained on; a new, unseen dataset would get no signal from those columns at all. Properties (size, dimensionality, task type, class imbalance) can generalize to datasets never seen during training.

Source: `pmlb.csv` — this file was listed in this notebook's original "Files Used in This Project" table back in `01a_butter_e_eda.ipynb` but never actually downloaded (confirmed empty in the raw data directory when this update started). Sourced here from PMLB's own official summary-statistics file (`EpistasisLab/pmlb`, `pmlb/all_summary_stats.tsv`) — all 12 of BUTTER-E's dataset names matched with 0 missing, 0 nulls in the columns used. **Naming note:** PMLB's actual column names are `n_instances` (not `n_observations`) and `imbalance`/`task` (lowercase, not `Imbalance`/`Task`) — used as-is from the source, renamed to the requested names in the output feature table.

Five new numeric auxiliary columns replace the 12 one-hot `dataset_*` columns: `n_observations`, `n_features`, `n_classes`, `task_encoded` (0=regression, 1=classification), `imbalance`. Still BUTTER-E-only, still masked to zero for EC-NAS per Section 3.3.2 — EC-NAS has no training-dataset-complexity equivalent (all CNN architectures here train on CIFAR-10 only, no per-row dataset variation to speak of).

**Design flag, resolved in `03a_within_butter_e.ipynb`'s closing note (still not wired into `02c`/the pooled pipeline until this update):** all of the above are BUTTER-E-only auxiliary columns, per the thesis's Section 3.3.2 design (masked to zero when not applicable to a family) rather than the shared 9-column core set from Section 3.3.1 — needed so RQ1's cross-family test sees a consistent input schema in both directions.

In [1]:
# IMPORTS

import pandas as pd

In [ ]:
# LOAD SOURCE DATA

DATA_PATH = "../../data/raw/butter_e/"

df = pd.read_csv(DATA_PATH + "runs_with_standardized_energy.csv")

# sanity check: this file should already be the filter-passed subset
assert (~df['filter']).all(), "unexpected filter==True rows found — filtering needed before feature build"

pmlb = pd.read_csv(DATA_PATH + "pmlb.csv")

df.shape, pmlb.shape

In [ ]:
# BUILD SHARED-SCHEMA FEATURE TABLE (core columns)

features = pd.DataFrame({
    'run_id': df['run_id'],
    'params': df['size'],
    'depth': df['depth'],
    'flops': 2 * df['size'],           # approximation: 2 FLOPs per parameter per forward pass
    'epochs': 3000,                     # fixed primary-sweep budget, see notebook intro
    'batch_size': df['batch_size'],
    'target': df['std_energy'],         # idle-power-corrected energy, in joules
    'family': 'MLP',
    'source_dataset': 'BUTTER-E',
})

features.shape

In [ ]:
# ADD is_gpu AND ONE-HOT shape — family-specific auxiliary columns, MLP-only for now
# (see the design flag in the intro markdown re: core vs. auxiliary schema)

features['is_gpu'] = df['is_gpu']

shape_dummies = pd.get_dummies(df['shape'], prefix='shape').astype(int)
features = pd.concat([features, shape_dummies], axis=1)

print(features.shape)
print([c for c in features.columns if c.startswith('shape_')])

In [ ]:
# ADD DATASET PROPERTIES (not dataset identity) — join pmlb.csv on df['dataset'].
# Replaces the one-hot dataset_* columns: properties generalize to datasets never seen
# during training, identity one-hot columns don't.

pmlb_join = df[['dataset']].merge(
    pmlb[['dataset', 'n_instances', 'n_features', 'n_classes', 'task', 'imbalance']],
    on='dataset', how='left',
)
assert pmlb_join.notnull().all().all(), "unmatched dataset(s) found in pmlb.csv"

features['n_observations'] = pmlb_join['n_instances'].to_numpy()
features['n_features'] = pmlb_join['n_features'].to_numpy()
features['n_classes'] = pmlb_join['n_classes'].to_numpy()
features['task_encoded'] = (pmlb_join['task'] == 'classification').astype(int).to_numpy()  # 0=regression, 1=classification
features['imbalance'] = pmlb_join['imbalance'].to_numpy()

print(features.shape)
print(features[['n_observations', 'n_features', 'n_classes', 'task_encoded', 'imbalance']].describe())

In [4]:
# SAVE

OUT_PATH = "../../data/processed/butter_e/butter_e_features.csv"
features.to_csv(OUT_PATH, index=False)
OUT_PATH

'../../data/processed/butter_e/butter_e_features.csv'

In [ ]:
# SANITY CHECK

print(features.shape)
print(features.columns.tolist())
print("nulls:", features.isnull().sum().sum())
features.head()